In [11]:
import numpy as np
import json
import torch
from torchsummary import summary
from pathlib import Path

In [12]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version used by PyTorch: {torch.version.cuda}")
else:
    print("CUDA is not available. PyTorch is using CPU.")

CUDA is not available. PyTorch is using CPU.


In [13]:
from src.models.base_model import base_model

Пути к каталогам.

In [14]:
config_dir = Path("./src/config/")
model_name = 'base-model'

config_path = config_dir / "config.json"
assert config_path.exists(), f"Config not found: {config_path}"
with open(config_path, "r") as f:
    general_config = json.load(f)

model_config_path = config_dir / f"{model_name}-config.json"
assert model_config_path.exists(), f"Config not found: {model_config_path}"
with open(model_config_path, "r") as f:
    model_config = json.load(f)

dataset_config_path = config_dir / f"{model_config['dataset_name']}-config.json"
assert dataset_config_path.exists(), f"Config not found: {dataset_config_path}"
with open(dataset_config_path, "r") as f:
    dataset_config = json.load(f)
    
data_path = Path(general_config['data_dir']) / dataset_config['dataset_name']

In [15]:
device = torch.device(general_config["device"].lower() if torch.cuda.is_available() else 'cpu')

# Тест архитектуры

In [16]:
mdl_input_size = model_config['input_size']

model = base_model(
    in_channels = mdl_input_size[0],
    out_channels = 4,
    features = model_config['feature_list'],
    #device = self.device
    )
model = model.to(device)
model.eval()

test_input = torch.randn(1, *mdl_input_size).to(device)
test_output = model(test_input)

model_size = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model size: {model_size}")
print(f"  Input:  {test_input.shape}")
print(f"  Output: {test_output.shape}")
summary(model, tuple(mdl_input_size))

Encoder features by level: [512, 1024]
Model size: 1058308
  Input:  torch.Size([1, 2, 512])
  Output: torch.Size([1, 4, 512])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1             [-1, 512, 512]           3,584
       BatchNorm1d-2             [-1, 512, 512]           1,024
              ReLU-3             [-1, 512, 512]               0
         MaxPool1d-4             [-1, 512, 256]               0
   ConvTranspose1d-5            [-1, 1024, 512]       1,049,600
            Conv1d-6               [-1, 4, 512]           4,100
Total params: 1,058,308
Trainable params: 1,058,308
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 11.02
Params size (MB): 4.04
Estimated Total Size (MB): 15.06
----------------------------------------------------------------


In [17]:
test_output>0.5

tensor([[[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]]])

## dataloader

In [18]:
SEED = np.random.randint(0, 10000)
torch.manual_seed(SEED)
np.random.seed(SEED)

In [19]:
from src.dataloaders.ZerosPolesDataset import TransformsConfig, ZerosPolesDataset

custom_transforms = TransformsConfig(
    #crop_ratio=[1.0, 1.0],
    time_delay=[0.0, 1e-9],
    noise_level=[5e-3, 30e-3],
    noise_reduce=3,
    gain=[-1e3, 1e3]
)

dataset = ZerosPolesDataset(
    dataset_dir = Path(general_config['data_dir']) / dataset_config['dataset_name'],
    split = 'train',
    transforms=custom_transforms
)

In [20]:
from src.utils.metrics import AverageMeter, CombinedLoss, DiceLoss, dice_coefficient, iou_score, pixel_accuracy




In [21]:
t1 = torch.tensor([[0, 1.0, 0], [1.0, 0, 1]], dtype=float)
t2 = torch.tensor([[1, 1.0, 0], [0, 1.0, 1]], dtype=float)

logits = torch.stack([t1, t2], dim=0)

t1 = torch.tensor([[0, 1, 0], [1, 0, 1]], dtype=float)
t2 = torch.tensor([[0, 1, 1], [0, 1, 1]], dtype=float)

targets = torch.stack([t1, t2], dim=0)

loss_func = torch.nn.BCEWithLogitsLoss()


In [27]:
dice_coefficient(logits, targets)

0.875

In [29]:
iou_score(logits, targets)

0.8333333730697632

In [23]:
np.mean([0.6931, 0.3133, 0.6931])

np.float64(0.5665000000000001)

In [24]:
y = torch.tensor(0.0)
p = torch.tensor(0.1)
-(y*torch.log(torch.sigmoid(p)) + (1-y)*torch.log(1-torch.sigmoid(p)))

tensor(0.7444)